In [ ]:
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go


DATA_DIR = Path("data")
VECTOR_FIELDS = {"velocity", "meanvelocity", "force"}
VECTOR_LABELS = {
    "velocity": ["u", "v", "w"],
    "meanvelocity": ["mean u", "mean v", "mean w"],
    "force": ["Fx", "Fy", "Fz"],
}
TENSOR_LABELS = ["xx", "yy", "zz", "xy", "xz", "yz"]


def load_field(path):
    with h5py.File(path, "r") as data_file:
        dataset_name = next(
            name for name in data_file if name not in {"xcoor", "ycoor", "zcoor", "values"}
        )
        field = np.asarray(data_file[dataset_name])
        coordinates = tuple(np.asarray(data_file[name]) for name in ("xcoor", "ycoor", "zcoor"))
    return dataset_name.removesuffix("_0001"), field, coordinates


fields = {}
for path in sorted(DATA_DIR.glob("*.h5")):
    name, field, coordinates = load_field(path)
    fields[name] = {"data": field, "coordinates": coordinates, "path": path}

print("Geladene Felder:")
for name, entry in fields.items():
    print(f"  {name}: {entry['data'].shape}")

# Alle vorhandenen Komponenten bzw. Beträge als 2D-Schnitte in mittlerer z-Hoehe.
plot_fields = {}
for name, entry in fields.items():
    field = entry["data"]
    if field.shape[-1] == 3 and name in VECTOR_FIELDS:
        for component, label in enumerate(VECTOR_LABELS[name]):
            plot_fields[f"{name} {label}"] = (field[..., component], entry["coordinates"])
        plot_fields[f"{name} |v|" if "velocity" in name else f"{name} |F|"] = (
            np.linalg.norm(field, axis=-1),
            entry["coordinates"],
        )
    elif field.shape[-1] == 6:
        for component, label in enumerate(TENSOR_LABELS):
            plot_fields[f"{name} {label}"] = (field[..., component], entry["coordinates"])
    else:
        plot_fields[name] = (np.squeeze(field, axis=-1), entry["coordinates"])

figure_columns = 4
figure_rows = int(np.ceil(len(plot_fields) / figure_columns))
fig_overview, axes = plt.subplots(
    figure_rows,
    figure_columns,
    figsize=(5 * figure_columns, 4 * figure_rows),
    squeeze=False,
    constrained_layout=True,
)

for axis, (label, (field, coordinates)) in zip(axes.flat, plot_fields.items()):
    x, y, z = coordinates
    z_index = len(z) // 2
    z_index = 18
    image = axis.contourf(x, y, field[z_index], levels=25, cmap="viridis")
    axis.set_title(f"{label}, z = {z[z_index]:.1f}")
    axis.set_xlabel("x")
    axis.set_ylabel("y")
    axis.set_aspect("equal")
    fig_overview.colorbar(image, ax=axis, label=label)

for axis in axes.flat[len(plot_fields):]:
    axis.set_visible(False)

fig_overview.savefig("all_fields_2d.png", dpi=180)
plt.show()


In [ ]:
# Betrag des vollständigen Geschwindigkeitsvektors |v| = sqrt(u^2 + v^2 + w^2).
speed_xyz = np.transpose(speed, (2, 1, 0))
speed_min, speed_max = np.nanpercentile(speed_xyz, [5, 95])

fig_speed = go.Figure(
    data=go.Isosurface(
        x=X_3d.ravel(),
        y=Y_3d.ravel(),
        z=Z_3d.ravel(),
        value=speed_xyz.ravel(),
        isomin=float(speed_min),
        isomax=float(speed_max),
        colorscale="Viridis",
        opacity=0.55,
        surface_count=8,
        caps=dict(x_show=False, y_show=False, z_show=False),
        colorbar=dict(title="|v| [m/s]"),
    )
)
fig_speed.update_layout(
    title="3D-Isosurface des Betrags des Geschwindigkeitsvektors |v|",
    width=1100,
    height=750,
    margin=dict(l=0, r=0, t=70, b=0),
    scene=dict(
        xaxis_title="x [m]",
        yaxis_title="y [m]",
        zaxis_title="z [m]",
        aspectmode="data",
    ),
)
fig_speed.write_html("velocity_3d_speed_magnitude.html", include_plotlyjs="cdn")
fig_speed.show()


In [ ]:
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np

DATA_DIR = Path("data")

with h5py.File(DATA_DIR/"meanvelocity.h5","r") as data_file:
    print(data_file.keys())
    field = np.asarray(data_file["meanvelocity_0001"])      # u, v, w (25, 30, 301, 3)  
    coordinates = tuple(np.asarray(data_file[name]) for name in ("xcoor", "ycoor", "zcoor"))        # x, y, z np_arrays (301, 30, 25)      -> 225.750 points

